# Clustering Alcohol Features

This analysis clusters haunted locations based on alcohol-related deaths, reported supernatural entities, and the time of reported hauntings. The features used for clustering are:

* Total_Deaths: The total number of alcohol-related deaths in the area.
* Percent_Under_21: The percentage of alcohol-related deaths involving individuals under the age of 21.
* Apparition_Type: The type of supernatural entity reported (e.g., ghost, demon, spirit, orb).
* Time_of_Day: The time when the supernatural event was reported (e.g., dusk, evening, morning, unknown).

By analyzing these factors, this clustering aims to investigate whether areas with higher alcohol-related deaths report more supernatural activity and whether certain apparition types and haunting times are more commonly associated with areas where alcohol-related mortality is higher, particularly among younger individuals.

# Generating Indices

The indices were chosen to ensure a balanced and representative sample of haunted places based on alcohol-related deaths, apparition type, and daylight duration. First, 500 haunted places in states with the highest alcohol-related deaths and 500 in states with the lowest alcohol-related deaths were selected. This selection was based on "Total_Deaths" and "Percent_Under_21" to compare supernatural activity in areas with varying levels of alcohol-related mortality.

Next, a diverse sample of up to 100 hauntings per apparition type was included to analyze how different supernatural experiences may correlate with alcohol-related deaths. This step ensures that a broad range of apparitions—such as ghosts, demons, spirits, and orbs—are fairly represented in the dataset. Additionally, 500 haunted places were selected based on daylight duration, with 250 locations experiencing the lowest daylight duration (<9.5 hours) and 250 with the highest daylight duration (>10.5 hours) to investigate whether lighting conditions influence paranormal reports.

In [69]:
# Number of rows to select per extreme group
num_rows_per_group = 500  

# Select haunted places in states with the HIGHEST alcohol-related deaths
high_alcohol_deaths = df.sort_values(by=['Total_Deaths', 'Percent_Under_21'], ascending=[False, False]).head(num_rows_per_group)

# Select haunted places in states with the LOWEST alcohol-related deaths
low_alcohol_deaths = df.sort_values(by=['Total_Deaths', 'Percent_Under_21'], ascending=[True, True]).head(num_rows_per_group)

# Get the index numbers of these rows
high_indices = high_alcohol_deaths.index.tolist()
low_indices = low_alcohol_deaths.index.tolist()

# Combine both sets of indices
selected_indices = high_indices + low_indices

# Select a subset of hauntings by apparition type
selected_hauntings = df.loc[selected_indices].groupby('Apparition_Type').apply(lambda x: x.sample(n=min(100, len(x)), random_state=42))

# Get the indices of selected hauntings by apparition type
apparition_indices = selected_hauntings.index.get_level_values(1).tolist()

# Select haunted places with the LOWEST daylight duration
low_light = df[df['Daylight_Duration_Hours'] < 9.5].sample(250, random_state=42)

# Select haunted places with the HIGHEST daylight duration
high_light = df[df['Daylight_Duration_Hours'] > 10.5].sample(250, random_state=42)

# Get the index numbers of these rows
low_light_indices = low_light.index.tolist()
high_light_indices = high_light.index.tolist()

# Combine all selected indices
final_selected_indices = apparition_indices + low_light_indices + high_light_indices

# # Print the final indices
# print(final_selected_indices)

# Jaccard

In [103]:
import pandas as pd
import os
import sys 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/jaccard/alcohol/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Jaccard Clusters

In [108]:
df.groupby('Cluster')[['Total_Deaths', 'Percent_Under_21']].describe()

Total_Deaths                    Percent_Under_21                   
                 count unique    top freq            count unique    top freq
Cluster                                                                      
cluster 0           84     30    493    8               84     20  2.60%   11
cluster 1          335     44  5,703   27              335     23  2.60%   40
cluster 2          390     48    526   40              390     24  2.30%   53

In [110]:
df.groupby(['Cluster', 'Apparition_Type'])['Time_of_Day'].value_counts(normalize=True).unstack()

Time_of_Day                                Dusk   Evening   Morning   Unknown
Cluster   Apparition_Type                                                    
cluster 0 Demon                             NaN  1.000000       NaN       NaN
          Demon, Ghost, Orb                 NaN       NaN       NaN  1.000000
          Evil Presence                     NaN  1.000000       NaN       NaN
          Female Ghost, Ghost               NaN       NaN       NaN  1.000000
          Female Ghost, Ghost, Spirit       NaN       NaN       NaN  1.000000
          Ghost                             NaN  0.500000       NaN  0.500000
          Ghost, Male Ghost                 NaN       NaN       NaN  1.000000
          Ghost, Orb                        NaN       NaN       NaN  1.000000
          Ghost, Phantom                    NaN       NaN       NaN  1.000000
          Ghost, Spirit                     NaN  0.666667       NaN  0.333333
          Orb                               NaN  0.333333       NaN  0.666667
          Phantom                           NaN  0.333333  0.333333  0.333333
          Spirit                            NaN       NaN  0.250000  0.750000
          Unknown                           NaN  0.227273  0.022727  0.750000
cluster 1 Demon                             NaN       NaN       NaN  1.000000
          Demon, Ghost, Orb                 NaN       NaN       NaN  1.000000
          Evil Presence                     NaN  0.333333       NaN  0.666667
          Female Ghost, Ghost               NaN  0.333333       NaN  0.666667
          Ghost                        0.040816  0.265306  0.061224  0.632653
          Ghost, Orb                        NaN       NaN       NaN  1.000000
          Ghost, Phantom                    NaN       NaN       NaN  1.000000
          Ghost, Spirit                     NaN  0.318182       NaN  0.681818
          Orb                               NaN  0.200000       NaN  0.800000
          Phantom                           NaN  0.400000       NaN  0.600000
          Spirit                            NaN  0.250000  0.083333  0.666667
          Unknown                           NaN  0.345550  0.026178  0.628272
          Demon, Spirit                     NaN       NaN       NaN  1.000000
          Evil Presence, Ghost              NaN  1.000000       NaN       NaN
          Floating Light                    NaN  0.500000       NaN  0.500000
          Floating Light, Ghost             NaN  1.000000       NaN       NaN
cluster 2 Ghost                             NaN  0.271318  0.054264  0.674419
          Orb                               NaN  0.250000       NaN  0.750000
          Phantom                           NaN  0.500000       NaN  0.500000
          Spirit                            NaN  0.265306  0.061224  0.673469
          Unknown                           NaN  0.290000  0.040000  0.670000

In [112]:
df.pivot_table(index='Cluster', columns='Apparition_Type', values='Daylight_Duration_Hours', aggfunc='mean')

Apparition_Type,Demon,"Demon, Ghost, Orb","Demon, Spirit",Evil Presence,"Evil Presence, Ghost","Female Ghost, Ghost","Female Ghost, Ghost, Spirit",Floating Light,"Floating Light, Ghost",Ghost,"Ghost, Male Ghost","Ghost, Orb","Ghost, Phantom","Ghost, Spirit",Orb,Phantom,Spirit,Unknown
Cluster,,,,,,,,,,,,,,,,,,
cluster 0,14.067,9.523,NaN,9.2555,NaN,9.046000,9.422,NaN,NaN,9.849500,9.265,9.2275,10.376000,9.163000,9.416000,12.064333,9.618500,11.087977
cluster 1,10.515,11.242,11.723,10.7530,9.478,8.706333,NaN,9.1625,9.385,9.996735,NaN,9.2428,9.100667,9.833818,9.441100,9.277000,10.139056,10.812419
cluster 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.095806,NaN,NaN,NaN,NaN,10.996625,9.368250,9.289837,10.335372


# Edit Distance

In [115]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/edit-distance/alcohol/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Edit Distance Clusters

In [118]:
df.groupby('Cluster')[['Total_Deaths', 'Percent_Under_21']].describe()

Total_Deaths                     Percent_Under_21                   
                 count unique     top freq            count unique    top freq
Cluster                                                                       
cluster 0           45     25   5,703    6               45     17  2.70%    6
cluster 1          171     32  10,647   24              171     21  4.00%   24
cluster 2          130     34     720   12              130     21  2.30%   15
cluster 3          412     46     526   42              412     23  2.60%   57
cluster 4            9      9   2,186    1                9      8  2.00%    2
cluster 5           42     17     493    7               42     13  1.40%    7

In [120]:
df.groupby(['Cluster', 'Apparition_Type'])['Time_of_Day'].value_counts(normalize=True).unstack()

Time_of_Day                                Dusk   Evening   Morning   Unknown
Cluster   Apparition_Type                                                    
cluster 0 Ghost, Orb                        NaN       NaN       NaN  1.000000
          Orb                               NaN       NaN       NaN  1.000000
          Phantom                           NaN       NaN  0.500000  0.500000
          Spirit                            NaN       NaN       NaN  1.000000
          Unknown                           NaN  0.187500  0.062500  0.750000
cluster 1 Ghost, Orb                        NaN       NaN       NaN  1.000000
          Orb                               NaN  0.333333       NaN  0.666667
          Phantom                           NaN  0.500000       NaN  0.500000
          Spirit                            NaN  0.333333  0.066667  0.600000
          Unknown                           NaN  0.388889  0.023810  0.587302
          Demon                             NaN       NaN       NaN  1.000000
          Ghost                        0.133333  0.400000  0.066667  0.400000
cluster 2 Orb                               NaN  0.333333       NaN  0.666667
          Spirit                            NaN  0.285714  0.035714  0.678571
          Unknown                           NaN  0.304348  0.065217  0.630435
          Ghost                             NaN  0.260000  0.080000  0.660000
cluster 3 Orb                               NaN  0.200000       NaN  0.800000
          Phantom                           NaN  0.500000       NaN  0.500000
          Spirit                            NaN  0.200000  0.133333  0.666667
          Unknown                           NaN  0.278261  0.026087  0.695652
          Demon                             NaN  1.000000       NaN       NaN
          Ghost                             NaN  0.280000  0.040000  0.680000
cluster 4 Demon, Ghost, Orb                 NaN       NaN       NaN  1.000000
          Demon, Spirit                     NaN       NaN       NaN  1.000000
          Evil Presence                     NaN       NaN       NaN  1.000000
          Evil Presence, Ghost              NaN  1.000000       NaN       NaN
          Female Ghost, Ghost               NaN       NaN       NaN  1.000000
          Floating Light                    NaN       NaN       NaN  1.000000
          Ghost, Phantom                    NaN       NaN       NaN  1.000000
          Ghost, Spirit                     NaN  0.500000       NaN  0.500000
cluster 5 Ghost, Orb                        NaN       NaN       NaN  1.000000
          Unknown                           NaN  1.000000       NaN       NaN
          Demon, Ghost, Orb                 NaN       NaN       NaN  1.000000
          Demon, Spirit                     NaN       NaN       NaN  1.000000
          Evil Presence                     NaN  0.750000       NaN  0.250000
          Female Ghost, Ghost               NaN  0.250000       NaN  0.750000
          Floating Light                    NaN  1.000000       NaN       NaN
          Ghost, Phantom                    NaN       NaN       NaN  1.000000
          Ghost, Spirit                     NaN  0.347826       NaN  0.652174
          Female Ghost, Ghost, Spirit       NaN       NaN       NaN  1.000000
          Floating Light, Ghost             NaN  1.000000       NaN       NaN
          Ghost, Male Ghost                 NaN       NaN       NaN  1.000000

In [122]:
df.pivot_table(index='Cluster', columns='Apparition_Type', values='Daylight_Duration_Hours', aggfunc='mean')

Apparition_Type,Demon,"Demon, Ghost, Orb","Demon, Spirit",Evil Presence,"Evil Presence, Ghost","Female Ghost, Ghost","Female Ghost, Ghost, Spirit",Floating Light,"Floating Light, Ghost",Ghost,"Ghost, Male Ghost","Ghost, Orb","Ghost, Phantom","Ghost, Spirit",Orb,Phantom,Spirit,Unknown
Cluster,,,,,,,,,,,,,,,,,,
cluster 0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.263,NaN,NaN,9.256750,10.40900,10.650200,10.867125
cluster 1,10.515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.854800,NaN,9.221,NaN,NaN,9.411167,10.25675,11.237867,11.055198
cluster 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.607540,NaN,NaN,NaN,NaN,9.749333,NaN,9.081893,10.258217
cluster 3,14.067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.137552,NaN,NaN,NaN,NaN,11.728400,9.70100,9.356533,10.426830
cluster 4,NaN,9.523,14.235,9.33900,9.478,9.12800,NaN,9.012,NaN,NaN,NaN,NaN,8.94500,12.498000,NaN,NaN,NaN,NaN
cluster 5,NaN,11.242,9.211,10.35775,NaN,8.77075,9.422,9.313,9.385,NaN,9.265,9.259,9.77725,9.514652,NaN,NaN,NaN,9.457000


# Cosine Similarity

In [126]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/cosine/alcohol/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Cosine Clusters

In [129]:
df.groupby('Cluster')[['Total_Deaths', 'Percent_Under_21']].describe()

Total_Deaths                    Percent_Under_21                   
                 count unique    top freq            count unique    top freq
Cluster                                                                      
cluster 0          809     49  5,703   60              809     25  2.60%  100

In [131]:
df.groupby(['Cluster', 'Apparition_Type'])['Time_of_Day'].value_counts(normalize=True).unstack()

Time_of_Day                                Dusk   Evening   Morning   Unknown
Cluster   Apparition_Type                                                    
cluster 0 Demon                             NaN  0.500000       NaN  0.500000
          Demon, Ghost, Orb                 NaN       NaN       NaN  1.000000
          Demon, Spirit                     NaN       NaN       NaN  1.000000
          Evil Presence                     NaN  0.600000       NaN  0.400000
          Evil Presence, Ghost              NaN  1.000000       NaN       NaN
          Female Ghost, Ghost               NaN  0.200000       NaN  0.800000
          Female Ghost, Ghost, Spirit       NaN       NaN       NaN  1.000000
          Floating Light                    NaN  0.500000       NaN  0.500000
          Floating Light, Ghost             NaN  1.000000       NaN       NaN
          Ghost                        0.010526  0.284211  0.052632  0.652632
          Ghost, Male Ghost                 NaN       NaN       NaN  1.000000
          Ghost, Orb                        NaN       NaN       NaN  1.000000
          Ghost, Phantom                    NaN       NaN       NaN  1.000000
          Ghost, Spirit                     NaN  0.360000       NaN  0.640000
          Orb                               NaN  0.238095       NaN  0.761905
          Phantom                           NaN  0.416667  0.083333  0.500000
          Spirit                            NaN  0.236559  0.086022  0.677419
          Unknown                           NaN  0.308046  0.032184  0.659770

In [133]:
df.pivot_table(index='Cluster', columns='Apparition_Type', values='Daylight_Duration_Hours', aggfunc='mean')

Apparition_Type,Demon,"Demon, Ghost, Orb","Demon, Spirit",Evil Presence,"Evil Presence, Ghost","Female Ghost, Ghost","Female Ghost, Ghost, Spirit",Floating Light,"Floating Light, Ghost",Ghost,"Ghost, Male Ghost","Ghost, Orb","Ghost, Phantom","Ghost, Spirit",Orb,Phantom,Spirit,Unknown
Cluster,,,,,,,,,,,,,,,,,,
cluster 0,12.291,10.3825,11.723,10.154,9.478,8.8422,9.422,9.1625,9.385,10.0547,9.265,9.238429,9.6108,9.75332,10.030095,10.00425,9.646839,10.621618
